# Tech Skills Brasil — análise diária

Este notebook apresenta as habilidades pedidas em vagas de entrada na área de tecnologia no Brasil. Ele usa sempre a versão disponível do dataset **Tech Skills Brasil** vinculada ao notebook.

A base considera vagas de estágio, trainee, aprendiz e nível júnior. Os resultados representam os anúncios encontrados durante o período da pesquisa.

In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

CAMINHOS_PREFERIDOS = (
    Path("/kaggle/input/tech-skills-br/vagas.parquet"),
    Path("/kaggle/input/datasets/rafaeldiasgarcia/tech-skills-br/vagas.parquet"),
)
CAMINHO_DADOS = next(
    (caminho for caminho in CAMINHOS_PREFERIDOS if caminho.exists()),
    CAMINHOS_PREFERIDOS[0],
)
FUSO_BRASIL = ZoneInfo("America/Sao_Paulo")
COR_FUNDO = "#11161d"
COR_TEXTO = "#d8e1ea"
COR_AZUL = "#4da3ff"
COR_VERDE = "#43c463"

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 30)

if not CAMINHO_DADOS.exists():
    encontrados = sorted(Path("/kaggle/input").rglob("vagas.parquet"))
    candidatos = [caminho for caminho in encontrados if "tech-skills-br" in caminho.parts]
    if len(candidatos) == 1:
        CAMINHO_DADOS = candidatos[0]
    elif len(encontrados) == 1:
        CAMINHO_DADOS = encontrados[0]
    else:
        raise FileNotFoundError(
            "Não foi possível localizar um único vagas.parquet. "
            f"Arquivos encontrados: {[str(caminho) for caminho in encontrados]}"
        )

display(Markdown(f"**Arquivo usado:** `{CAMINHO_DADOS}`"))

In [ ]:
vagas = pd.read_parquet(CAMINHO_DADOS)
colunas_obrigatorias = {
    "source", "external_id", "title", "company", "skills",
    "area", "seniority", "workplace_type", "published_date", "url",
}
faltantes = sorted(colunas_obrigatorias - set(vagas.columns))
if faltantes:
    raise ValueError(f"O dataset não contém as colunas obrigatórias: {faltantes}")

vagas["published_date"] = pd.to_datetime(vagas["published_date"], errors="coerce")
tem_skills = vagas["skills"].fillna("").str.strip().ne("")
agora = datetime.now(FUSO_BRASIL)

resumo = pd.DataFrame(
    {
        "Indicador": [
            "Vagas na base",
            "Vagas com habilidades",
            "Empresas",
            "Fontes",
        ],
        "Valor": [
            len(vagas),
            int(tem_skills.sum()),
            vagas["company"].nunique(dropna=True),
            vagas["source"].nunique(dropna=True),
        ],
    }
)
display(resumo.style.hide(axis="index").format({"Valor": "{:,.0f}"}))

## Ranking de habilidades

Uma vaga pode citar mais de uma habilidade. Por isso, esta seção separa os valores da coluna `skills` antes de calcular o ranking.

In [ ]:
colunas_contexto = [
    "source", "external_id", "title", "company", "area",
    "seniority", "workplace_type", "location", "regiao",
    "polo", "published_date", "url", "skills",
]
analise_skills = vagas.loc[tem_skills, colunas_contexto].copy()
analise_skills["skill"] = analise_skills.pop("skills").str.split(";")
analise_skills = analise_skills.explode("skill", ignore_index=True)
analise_skills["skill"] = analise_skills["skill"].str.strip()
analise_skills = analise_skills[analise_skills["skill"].ne("")].copy()

ranking_skills = (
    analise_skills.groupby("skill", observed=True)
    .agg(vagas=("external_id", "size"), empresas=("company", "nunique"))
    .reset_index()
    .sort_values(["vagas", "skill"], ascending=[False, True], ignore_index=True)
)
ranking_skills.insert(0, "#", ranking_skills.index + 1)
base_demanda = max(int(tem_skills.sum()), 1)
ranking_skills["demanda_%"] = ranking_skills["vagas"].div(base_demanda).mul(100)

display(
    ranking_skills.head(30).style.hide(axis="index").format(
        {"vagas": "{:,.0f}", "empresas": "{:,.0f}", "demanda_%": "{:.1f}%"}
    )
)

In [ ]:
top_skills = ranking_skills.head(20).sort_values("vagas")
fig, ax = plt.subplots(figsize=(11, 8), facecolor=COR_FUNDO)
ax.set_facecolor(COR_FUNDO)
ax.barh(top_skills["skill"], top_skills["vagas"], color=COR_AZUL)
ax.set_title("20 habilidades mais citadas", color=COR_TEXTO, fontsize=15, pad=14)
ax.set_xlabel("Número de vagas", color=COR_TEXTO)
ax.tick_params(colors=COR_TEXTO)
for borda in ax.spines.values():
    borda.set_visible(False)
ax.grid(axis="x", color="#33404d", alpha=0.45)
plt.tight_layout()
plt.show()

## Distribuição das vagas

In [ ]:
distribuicao_areas = (
    vagas["area"].astype("string").fillna("Não informada").value_counts().rename_axis("área").reset_index(name="vagas")
)
distribuicao_areas["percentual"] = distribuicao_areas["vagas"].div(len(vagas)).mul(100)
display(
    distribuicao_areas.head(20).style.hide(axis="index").format(
        {"vagas": "{:,.0f}", "percentual": "{:.1f}%"}
    )
)

top_areas = distribuicao_areas.head(12).sort_values("vagas")
fig, ax = plt.subplots(figsize=(11, 6), facecolor=COR_FUNDO)
ax.set_facecolor(COR_FUNDO)
ax.barh(top_areas["área"], top_areas["vagas"], color=COR_VERDE)
ax.set_title("Distribuição por área técnica", color=COR_TEXTO, fontsize=15, pad=14)
ax.set_xlabel("Número de vagas", color=COR_TEXTO)
ax.tick_params(colors=COR_TEXTO)
for borda in ax.spines.values():
    borda.set_visible(False)
ax.grid(axis="x", color="#33404d", alpha=0.45)
plt.tight_layout()
plt.show()

In [ ]:
data_inicial = vagas["published_date"].min()
data_final = vagas["published_date"].max()
display(
    Markdown(
        f"**Período dos anúncios:** {data_inicial:%d/%m/%Y} a {data_final:%d/%m/%Y}  \
"
        f"**Notebook executado em:** {agora:%d/%m/%Y às %H:%M} (Brasília)  \
"
        f"**Menções de habilidades analisadas:** {len(analise_skills):,.0f}"
    )
)